In [1]:
from nba_api.stats.endpoints import ScheduleLeagueV2
from supabase import create_client, Client
import pandas as pd
import numpy as np
import requests
import json
import csv

In [2]:
df = ScheduleLeagueV2(season= "2025-26", league_id= "00")

games = df.get_data_frames()[0]
print(games.head())

  leagueId seasonYear             gameDate      gameId         gameCode  \
0       00    2025-26  10/02/2025 00:00:00  0012500008  20251002/PHINYK   
1       00    2025-26  10/03/2025 00:00:00  0012500001  20251003/PHXLAL   
2       00    2025-26  10/03/2025 00:00:00  0012500009  20251003/MELNOP   
3       00    2025-26  10/04/2025 00:00:00  0012500010  20251004/NYKPHI   
4       00    2025-26  10/04/2025 00:00:00  0012500011  20251004/SEMNOP   

   gameStatus        gameStatusText  gameSequence           gameDateEst  \
0           3                 Final             1  2025-10-02T00:00:00Z   
1           3                 Final             2  2025-10-03T00:00:00Z   
2           3  Final                            1  2025-10-03T00:00:00Z   
3           3                 Final             1  2025-10-04T00:00:00Z   
4           3  Final                            5  2025-10-04T00:00:00Z   

            gameTimeEst  ... awayOttBroadcasters_broadcasterMedia  \
0  1900-01-01T12:00:00Z  ... 

In [10]:
# FILTERING FOR REGULAR SEASON AND POSTSEASON
game_ids = games.loc[
    games["gameId"].astype(str).str.startswith(("002", "004")),
    "gameId"
]

In [13]:
game_ids = [
    "0022500004",
    "0022500082",
    "0022500083",
    "0022500084",
    "0022500085",
    "0022500086",
    "0022500087",
    "0022500088",
    "0022500089",
    "0022500147",
    "0022500244",
    "0022500255",
    "0022500256",
    "0022500257",
    "0022500258",
    "0022500259",
    "0022500260",
    "0022500261",
    "0022500062",
    "0022500296",
    "0022500422",
    "0022500009",
    "0022500010",
    "0022500537",
    "0022500716",
    "0022500717",
    "0022500718",
    "0022500719",
    "0022500720",
    "0022500721",
    "0022500722",
    "0022500723",
    "0022500724",
    "0022500725",
    "0022500867",
    "0022500651",
    "0022500652"
]

In [15]:
import requests
import pandas as pd
import time

url = "https://stats.nba.com/stats/shotqualityshotlog"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "x-nba-stats-origin": "stats",
    "x-nba-stats-token": "true",
    "Accept": "application/json",
}

all_shots = []
failed_games = []

for i, game_id in enumerate(game_ids):

    game_id = str(game_id)

    print(f"{i+1}/{len(game_ids)} - {game_id}")

    params = {
        "GameID": game_id,
        "LeagueID": "00"
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )

        print(response.status_code)

        if response.status_code != 200:
            print(response.text[:300])
            failed_games.append(game_id)
            continue

        data = response.json()

        shots = data.get("shots", [])

        if len(shots) == 0:
            print(f"No shots found for {game_id}")
            continue

        temp_df = pd.DataFrame(shots)
        temp_df["GAME_ID"] = game_id

        all_shots.append(temp_df)

        time.sleep(5)

    except Exception as e:
        print(f"Failed {game_id}: {e}")
        failed_games.append(game_id)

shot_quality_df = pd.concat(all_shots, ignore_index=True)

print("Final shape:", shot_quality_df.shape)
print("Failed games:", failed_games)

shot_quality_df.to_csv("all_shot_quality_shots2.csv", index=False)

1/37 - 0022500004
200
No shots found for 0022500004
2/37 - 0022500082
200
No shots found for 0022500082
3/37 - 0022500083
200
No shots found for 0022500083
4/37 - 0022500084
200
No shots found for 0022500084
5/37 - 0022500085
200
No shots found for 0022500085
6/37 - 0022500086
200
No shots found for 0022500086
7/37 - 0022500087
200
No shots found for 0022500087
8/37 - 0022500088
200
No shots found for 0022500088
9/37 - 0022500089
200
No shots found for 0022500089
10/37 - 0022500147
200
No shots found for 0022500147
11/37 - 0022500244
200
No shots found for 0022500244
12/37 - 0022500255
200
No shots found for 0022500255
13/37 - 0022500256
200
No shots found for 0022500256
14/37 - 0022500257
200
No shots found for 0022500257
15/37 - 0022500258
200
No shots found for 0022500258
16/37 - 0022500259
200
No shots found for 0022500259
17/37 - 0022500260
200
No shots found for 0022500260
18/37 - 0022500261
200
No shots found for 0022500261
19/37 - 0022500062
200
No shots found for 0022500062
20

ValueError: No objects to concatenate

In [8]:
import requests
import pandas as pd

url = "https://stats.nba.com/stats/shotqualityshotlog"

test_game_id = "0042500301"

params = {
    "GameID": test_game_id,
    "LeagueID": "00"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "x-nba-stats-origin": "stats",
    "x-nba-stats-token": "true",
    "Accept": "application/json",
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.text[:500])

data = response.json()

shots = data.get("shots", [])

test_df = pd.DataFrame(shots)

print(test_df.shape)
print(test_df.head())

200
{"parameters":{"leagueId":"00","gameDate":"2026-05-19","gameId":"0042500301"},"shots":[{"gameId":"0042500301","gameDate":"2026-05-19","matchup":"CLE @ NYK","homeTeamid":1610612752,"homeTeamCity":"New York","homeTeamName":"Knicks","homeTeamAbbreviation":"NYK","awayTeamid":1610612739,"awayTeamCity":"Cleveland","awayTeamName":"Cavaliers","awayTeamAbbreviation":"CLE","playerId":1628384,"playerName":"OG Anunoby","teamId":1610612752,"teamCity":"New York","teamName":"Knicks","teamAbbreviation":"NYK","s
(158, 37)
       gameId    gameDate    matchup  homeTeamid homeTeamCity homeTeamName  \
0  0042500301  2026-05-19  CLE @ NYK  1610612752     New York       Knicks   
1  0042500301  2026-05-19  CLE @ NYK  1610612752     New York       Knicks   
2  0042500301  2026-05-19  CLE @ NYK  1610612752     New York       Knicks   
3  0042500301  2026-05-19  CLE @ NYK  1610612752     New York       Knicks   
4  0042500301  2026-05-19  CLE @ NYK  1610612752     New York       Knicks   

  homeTeamAbbrev